<a href="https://colab.research.google.com/github/Amruth-U-tech/DL-Journey/blob/main/08-LLMs/LLM-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("using model:",model_name)
print("device:",device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


using model: bert-base-uncased
device: cuda


In [5]:
def show_mlm_predictions(sentence:str,top_k:int = 5):
  if tokenizer.mask_token not in sentence:
    raise ValueError(f"This tokenizer does not have a mask token: {tokenizer.mask_token}")

  inputs = tokenizer(sentence, return_tensors="pt")
  inputs = {k: v.cuda() for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)
  logits = outputs.logits

  mask_token_id = tokenizer.mask_token_id
  mask_positions = (inputs["input_ids"]==mask_token_id).nonzero(as_tuple=False)

  if mask_positions.numel()==0:
    raise ValueError(f"No mask token found in the input: {sentence}")

  _,mask_index = mask_positions[0]
  mask_index = mask_index.item()

  mask_logits = logits[0,mask_index,:]
  probs = torch.softmax(mask_logits,dim=-1)
  topk = torch.topk(probs,k=top_k)

  print(f"input sentence: {sentence}\n")
  print("top predictions for [mask]:")
  for rank in range(top_k):
    token_id = topk.indices[rank].item()
    token_str = tokenizer.decode([token_id]).strip()
    prob = topk.values[rank].item()
    print(f"{rank+1:>2}.{token_str:15s}  (prob = {prob:.3f})")


In [6]:
sentence = f"The capital of france is {tokenizer.mask_token}."
show_mlm_predictions(sentence,top_k=5)

input sentence: The capital of france is [MASK].

top predictions for [mask]:
 1.paris            (prob = 0.417)
 2.lille            (prob = 0.071)
 3.lyon             (prob = 0.063)
 4.marseille        (prob = 0.044)
 5.tours            (prob = 0.030)


In [10]:
sentence = f"there was a great king who was a great fighter and {tokenizer.mask_token}"
show_mlm_predictions(sentence,top_k=5)

input sentence: there was a great king who was a great fighter and [MASK]

top predictions for [mask]:
 1..                (prob = 0.745)
 2.!                (prob = 0.121)
 3.;                (prob = 0.109)
 4.?                (prob = 0.017)
 5.|                (prob = 0.005)


In [11]:
sentence = f"deep learning models are very {tokenizer.mask_token}"
show_mlm_predictions(sentence,top_k=8)

input sentence: deep learning models are very [MASK]

top predictions for [mask]:
 1..                (prob = 0.921)
 2.;                (prob = 0.045)
 3.?                (prob = 0.003)
 4.simple           (prob = 0.002)
 5.popular          (prob = 0.001)
 6.!                (prob = 0.001)
 7.effective        (prob = 0.001)
 8.basic            (prob = 0.001)
